<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 28px; border-radius: 10px; color: #0f172a; font-family: sans-serif;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Pipeline Stage 04
    </span>
    <h1 style="color: #0f172a; margin-top: 10px; margin-bottom: 8px; font-size: 26px; border-bottom: none;">
        Feature Engineering & Dataset Assembly
    </h1>
    <p style="color: #475569; font-size: 14px; margin-bottom: 20px;">
        Consolidates cleaned price action, fetches external market indicators, and generates cross-sectional technical features for global machine learning models.
    </p>
    <div style="background-color: #f1f5f9; padding: 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <p style="margin: 0; color: #1e40af; font-size: 12px; font-weight: bold; text-transform: uppercase;">Completed Steps in this Module:</p>
        <ol style="margin-top: 8px; margin-bottom: 0; padding-left: 20px; color: #334155; font-size: 13px; line-height: 1.6;">
            <li><b>Load Processed Datasets:</b> Consolidate asset classes, handle forward-fills, and align start dates.</li>
            <li><b>Fetch Yahoo Features:</b> Retrieve OHLCV, dividends, and stock splits for tradable assets.</li>
            <li><b>Merge Market Data:</b> Unify market data with synthetic assets into a single master schema.</li>
            <li><b>Technical Feature Engineering:</b> Generate vectorized returns, rolling volatility, drawdown, and volume features.</li>
        </ol>
    </div>
    <p style="margin-top: 15px; margin-bottom: 0; color: #64748b; font-size: 12px;">
        <b>Next Milestones:</b> Feature EDA & Redundancy Check &rarr; Forward Target Engineering (t+20) &rarr; Walk-Forward Model Training
    </p>
</div>

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import os
#os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
from pathlib import Path
sys.path.append(os.path.abspath(".."))

from src.data_loader import build_master_dataset
from src.fetch_features import fetch_yfinance_features
from src.feature_engineering import prepare_market_data, prepare_master_market_data

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Step 01
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Load Processed Datasets
    </h3>
    <p style="color: #475569; font-size: 13.5px; margin-bottom: 14px; line-height: 1.5;">
        Loads all asset classes (i.e only Adjused Close), strips weekends and holidays using the master stock trading calendar, merges into a unified DataFrame, and aligns to the common start date.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <p style="margin: 0; color: #1e40af; font-size: 12px; font-weight: 600; font-style: italic;">
            Note: Uses datasets previously generated in the <code>03_risk_analysis.ipynb</code> pipeline.
        </p>
    </div>
</div>

In [ ]:
master_df = build_master_dataset()

#DATA_DIR =Path("../data/features")
#output_path = Path(DATA_DIR) / ("verified_asset_data.csv")
#master_df.to_csv(output_path)

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Step 02
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Fetch Yahoo Finance Features
    </h3>
    <p style="color: #475569; font-size: 13.5px; margin-bottom: 14px; line-height: 1.5;">
        Downloads incremental market data for all asset classes (Stocks, ETFs, Crypto Currency, Commodities) handles currency conversion to EUR, fetches and appends up-to-date synthetic ETF series, and aligns all records onto a master stock business calendar while cleaning corporate action fields as per their asset class.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <p style="margin: 0; color: #1e40af; font-size: 12px; font-weight: 600; font-style: italic;">
            Artifact Generated: <code>yfinance_raw_features.parquet</code>
        </p>
    </div>
</div>

In [ ]:
yf_features = fetch_yfinance_features()
yf_features.head()

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Step 03
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Merge Market Data
    </h3>
    <p style="color: #475569; font-size: 13.5px; margin-bottom: 14px; line-height: 1.5;">
        Merges the Adjusted Close data (Bond, and Real estate) with Yahoo Finance features into one unified master dataset ready for feature computation.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <p style="margin: 0; color: #1e40af; font-size: 12px; font-weight: 600; font-style: italic;">
            Artifact Generated: <code>merged_market_dataset.parquet</code>
        </p>
    </div>
</div>

In [ ]:
market_df = prepare_market_data(master_df, yf_features)
market_df.head()

<div style="background-color: #f8fafc; border: 1px solid #e2e8f0; padding: 24px; border-radius: 10px; color: #0f172a; font-family: sans-serif; margin-bottom: 20px;">
    <span style="background-color: #2563eb; color: #ffffff; padding: 3px 10px; border-radius: 12px; font-size: 11px; font-weight: bold; text-transform: uppercase;">
        Step 04
    </span>
    <h3 style="color: #0f172a; margin-top: 8px; margin-bottom: 6px; font-size: 20px; border-bottom: none; font-weight: 700;">
        Technical Feature Engineering (Vectorized)
    </h3>
    <p style="color: #475569; font-size: 13.5px; margin-bottom: 14px; line-height: 1.5;">
        Computes grouped cross-sectional indicators: <b>Log Returns</b> (1, 5, 20, 60D), <b>Rolling Volatility</b>, <b>SMA/EMA Ratios</b>, <b>MACD</b>, <b>RSI-14</b>, <b>Bollinger Width</b>, zero-warmup <b>Max Drawdown (60D)</b>, and <b>OBV Z-Scores</b>.
    </p>
    <div style="background-color: #f1f5f9; padding: 12px 15px; border-radius: 8px; border-left: 4px solid #2563eb;">
        <p style="margin: 0; color: #1e40af; font-size: 12px; font-weight: 600; font-style: italic;">
            Artifact Generated: <code>master_dataset.parquet</code>
        </p>
    </div>
</div>

In [ ]:
feature_df = prepare_master_market_data(market_df)
feature_df.head()